# Hebrew Acronym Disambiguation: Local Dev Study

We compare five arms: trained **DictaBERT selection**, **Qwen generation and selection**, and **Gemini generation and selection**. Every arm receives the same identified occurrences with the exact target marked `[ACR]...[/ACR]`. The LLMs receive identical prompts per task; selection receives the same shuffled candidates. Generation receives neither candidates nor gold. This is a comparison of systems, not a causal test of model size or the contribution of context.

Preview uses invented examples and performs no model/service calls or research reads. Reload reads saved evidence only. Selection accuracy is preliminary dev analysis; generation requires human review. No actual checkpoint, service or research performance was validated while implementing this notebook.

## Local setup and manual validation

From this checkout, use an isolated Python environment. [requirements.txt](../requirements.txt) pins project dependencies. The original checkpoint additionally requires matching recorded `torch`, `transformers`, `tokenizers`, model and tokenizer content. Keep its JSON unchanged.

```sh
python3.12 -m venv .venv
.venv/bin/python -m pip install -e . jupyterlab ipykernel
.venv/bin/python -m pip check
.venv/bin/python -m jupyter lab notebooks/experimental_study.ipynb
```

Select that environment’s kernel. Fill `checkpoint` with Ben’s exact weights filename; the adjacent `<weights>.json` must be the original. If the saved base snapshot moved, set `snapshot_path` to a content-identical local copy. CPU is the explicit default; Colab is an alternative with compatible files and libraries.

**Qwen:** keep `qwen2.5:7b` through local Ollama. Check an already provisioned service/model with `ollama list` and `curl http://localhost:11434/api/tags`. No notebook code starts a service or pulls a model. Enabled requests record the server-reported digest, server version and settings.

**Gemini:** Shaked selected `gemini-3.8-flash` with `thinkingLevel="low"`. Low thinking is not disabled thinking. Set `GEMINI_API_KEY` only in the environment inherited by the notebook kernel; never paste it into this notebook, settings, saved output or Git. No `.env` file is loaded automatically. The key is sent in `x-goog-api-key`, not the URL. The request records `modelVersion`, completion reason and available usage; it does not claim digest verification. A missing or changed modelVersion makes the response an identity failure while retaining final text. Omitted settings use provider defaults. [Model and supported thinking levels](https://ai.google.dev/gemini-api/docs/models/gemini-3.8-flash), [response API](https://ai.google.dev/api/generate-content), [authentication](https://ai.google.dev/gemini-api/docs/api-key).

1. Use `mode="run"`, `run_kind="validation"`, and explicit paths. Enable each system independently. DictaBERT predicts three dev items; each enabled LLM sends **one generation and one selection request**. With both LLMs, this is four requests. Other selected items remain visibly unrun. Timeout is 120 seconds per request; there are no automatic retries.
2. After manual success, rerun all cells from fresh settings with `run_kind="full_dev"`. A new run ID separates the outputs. All 62 dev items are requested per enabled system: **248 LLM requests** with both providers and both tasks.
3. To inspect saved results, use `mode="reload"`, the exact `saved_run` JSON path and `saved_run_id`. No model, service or credential is required. Format-1 runs show only their original DictaBERT/Qwen arms; no Gemini results are invented. Saved encoder reuse requires identical input rows and checkpoint provenance.

Run all cells in order after changing settings. Raw outputs are saved outside Git after each attempt. A provider failure preserves the other provider’s results. Never rerun a completed arm in place; start a new run.

In [ ]:
from pathlib import Path
from uuid import uuid4
from IPython.display import HTML, display
from hebrew_acronyms import experimental_study as study
from hebrew_acronyms.models.dictabert_cross_encoder.model import inspect_checkpoint

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
settings = dict(
    mode="preview", run_kind="validation", run_id=str(uuid4()), root=ROOT,
    input_path=ROOT / "data/study_v1/encoder_inputs/dev.csv", expected_dev_items=62,
    validation_items=3, device="cpu", checkpoint=None, snapshot_path=None,
    enable_encoder=False, saved_encoder=None, saved_encoder_run_id=None,
    enable_qwen=False, qwen_model="qwen2.5:7b", ollama_url="http://localhost:11434",
    qwen_options={}, enable_gemini=False, gemini_model="gemini-3.8-flash",
    gemini_generation_config={"thinkingConfig": {"thinkingLevel": "low"}},
    request_timeout=120, output_root=ROOT.parent / "artifacts/study-runs",
    saved_run=None, saved_run_id=None,
)
checkpoint_info = None
if settings["mode"] == "run" and settings["enable_encoder"]:
    checkpoint_info = inspect_checkpoint(settings["checkpoint"], snapshot_path=settings["snapshot_path"])
checkpoint_info

## 1. Input and run identity

The qualified dev input has 62 items. Validation takes three items; preview uses invented examples. Complete rows, their order and IDs bind all five arms. Disabled, failed and unfinished arms retain a record for every selected item. Historical mixed/test inputs are never selected as fallbacks.

In [ ]:
rows, cohort = study.read_study_inputs(settings)
artifact = study.prepare_study(rows, cohort, settings)
run = artifact["settings"]
request_limit = 1 if run.get("run_kind") == "validation" else None
print("Run:", artifact["run_id"], "Cohort:", cohort)

## 2. DictaBERT selection

The existing loader verifies original checkpoint identities and any explicitly relocated snapshot. Prediction returns one candidate or failure per item and releases the encoder before LLM requests. No training occurs.

In [ ]:
encoder_predictions = None
if settings["mode"] == "run" and (run["enable_encoder"] or run["saved_encoder"]):
    encoder_predictions = study.predict_encoder(
        artifact, rows, checkpoint=run["checkpoint"], snapshot_path=run["snapshot_path"],
        device=run["device"], saved_path=run["saved_encoder"], saved_run_id=run["saved_encoder_run_id"],
    )

## 3. Qwen free expansion

Generation uses the marked sentence and raw target, including attached prefixes. It never sees gold or candidates. The bound callable supplies the same model and settings to both Qwen tasks.

In [ ]:
if settings["mode"] == "run" and run["enable_qwen"]:
    qwen = study.connect_qwen(artifact)
    study.collect_responses(artifact, "generate", qwen, system="qwen",
                            limit=request_limit, on_record=study.save_study)

## 4. Qwen candidate selection

Only one uppercase in-range letter after trimming is valid. Extra explanation, lowercase, multiple letters, empty answers and out-of-range letters fail parsing. The original raw response and shown order remain saved.

In [ ]:
if settings["mode"] == "run" and run["enable_qwen"]:
    study.collect_responses(artifact, "select", qwen, system="qwen",
                            limit=request_limit, on_record=study.save_study)

## 5. Gemini free expansion

Gemini receives the same generation prompt as Qwen. Final text parts are combined and thought parts are excluded. Blocking, missing text, timeout and incomplete completion are explicit failures; partial final text is retained for inspection.

In [ ]:
if settings["mode"] == "run" and run["enable_gemini"]:
    gemini = study.connect_gemini(artifact)
    study.collect_responses(artifact, "generate", gemini, system="gemini",
                            limit=request_limit, on_record=study.save_study)

## 6. Gemini candidate selection

The item’s prompt and candidate order match Qwen exactly, independent of provider success. Both Gemini tasks use the same requested model/configuration; returned versions are checked for consistency across the run.

In [ ]:
if settings["mode"] == "run" and run["enable_gemini"]:
    study.collect_responses(artifact, "select", gemini, system="gemini",
                            limit=request_limit, on_record=study.save_study)

## 7. Save raw evidence

Each attempt is saved atomically within its run. Model identities, request settings, exact prompts, raw responses and failures remain inspectable. Reload never rewrites historical files or calls current prompt builders.

In [ ]:
artifact_path = None
if settings["mode"] == "run":
    artifact_path = study.save_study(artifact)
print("Saved artifact:", artifact_path)

## 8. Inspect preliminary dev results

Selected candidates are compared to gold using exact equality after trimming, with no aliases or substring scoring. Micro divides correct predictions by every requested item. Macro is the unweighted mean of within-type accuracies. Failures and unrun items stay in the denominator of an attempted system; an entirely unrun system is labelled **not measured**. Missing type metadata makes macro unavailable. A validation subset is never labelled full dev.

The summary compares all selection systems present in the artifact. Expand an item to inspect original/marked context, gold, candidates, letter mappings, raw answers, decoded selections and failure states. Generation remains manual review. Disagreements compare valid selections across DictaBERT, Qwen and Gemini. Ten item panels are shown per page; change `start` to inspect later items. Full derived results remain in `results` without rewriting raw evidence.

In [ ]:
prediction_view, results = study.display_tables(artifact, start=0, max_items=10)
display(HTML(prediction_view))
metrics = results["metrics"]
disagreements = results["disagreements"]

### Limits and reproducibility

These are development measures, not test results or a final semantic judgment protocol. Different systems also differ in training, architecture and service behavior; this design does not isolate model size. Inspect `checkpoint_info`, `rows`, `artifact`, `metrics` and `disagreements` for intermediate evidence. Format-1 artifacts retain their original scope and provenance limitations.

Fixture tests establish code behavior, not actual model quality, checkpoint compatibility or account/service availability. The first manual validation run supplies that evidence. The [training appendix](train_dictabert.ipynb) remains separate. AI assistance contributed implementation and fixture checks, not human annotation or scientific validation.